# 02 - Preprocessing

This notebook:
- Loads raw data, cleans column names
- Encodes categorical variables (`GENDER`, `LUNG_CANCER`)
- Maps binary features from (1,2) to (0,1)
- Scales `AGE` using StandardScaler
- Performs stratified train/test split (80/20)
- Saves processed data and preprocessing objects for reuse

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1) Load raw data
DATA_PATH = Path('..') / 'data' / 'raw' / 'survey lung cancer.csv'
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (559, 16)


,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,M,69,1,2,2,1,1,2,1,2,2,2,2,2,2,YES
1,M,74,2,1,1,1,2,2,2,1,1,1,2,2,2,YES
2,F,59,1,1,1,2,1,2,1,2,1,2,2,1,2,NO
3,M,63,2,2,2,1,1,1,1,1,2,1,1,2,2,NO
4,F,63,1,2,1,1,1,1,1,2,1,2,2,1,1,NO


In [ ]:
# 2) Encode GENDER: M-1, F-0
df['GENDER'] = df['GENDER'].map({'M': 1, 'F': 0})

# 3) Encode LUNG_CANCER target: YES-1, NO-0
df['LUNG_CANCER'] = df['LUNG_CANCER'].map({'YES': 1, 'NO': 0})

# 4) Map binary features from (1,2) to(0,1)
binary_cols = [c for c in df.columns if c not in ['GENDER', 'AGE', 'LUNG_CANCER']]
for col in binary_cols:
    df[col] = df[col].map({1: 0, 2: 1})

print("After encoding:")
print(df.dtypes)
df.head()

After encoding:
GENDER                   int64
AGE                      int64
SMOKING                  int64
YELLOW_FINGERS           int64
ANXIETY                  int64
PEER_PRESSURE            int64
CHRONIC DISEASE          int64
FATIGUE                  int64
ALLERGY                  int64
WHEEZING                 int64
ALCOHOL CONSUMING        int64
COUGHING                 int64
SHORTNESS OF BREATH      int64
SWALLOWING DIFFICULTY    int64
CHEST PAIN               int64
LUNG_CANCER              int64
dtype: object


,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,1,69,0,1,1,0,0,1,0,1,1,1,1,1,1,1
1,1,74,1,0,0,0,1,1,1,0,0,0,1,1,1,1
2,0,59,0,0,0,1,0,1,0,1,0,1,1,0,1,0
3,1,63,1,1,1,0,0,0,0,0,1,0,0,1,1,0
4,0,63,0,1,0,0,0,0,0,1,0,1,1,0,0,0


In [4]:
# 5) Separate features and target
X = df.drop('LUNG_CANCER', axis=1)
y = df['LUNG_CANCER']
print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nTarget ratio (YES): {y.mean():.2%}")

Features shape: (559, 15)
Target distribution:
LUNG_CANCER
1    488
0     71
Name: count, dtype: int64

Target ratio (YES): 87.30%


In [5]:
# 6) Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist:\n{y_train.value_counts()}")
print(f"Test target dist:\n{y_test.value_counts()}")

Train: (447, 15), Test: (112, 15)
Train target dist:
LUNG_CANCER
1    390
0     57
Name: count, dtype: int64
Test target dist:
LUNG_CANCER
1    98
0    14
Name: count, dtype: int64


In [8]:
# 7) Scale AGE using StandardScaler
scaler = StandardScaler()
X_train['AGE'] = scaler.fit_transform(X_train[['AGE']])
X_test['AGE'] = scaler.transform(X_test[['AGE']])

print("AGE statistics after scaling:")
print(f"  Train mean: {X_train['AGE'].mean():.2f}, std: {X_train['AGE'].std():.2f}")
print(f"  Test mean:  {X_test['AGE'].mean():.2f}, std: {X_test['AGE'].std():.2f}")

AGE statistics after scaling:
  Train mean: 0.00, std: 1.00
  Test mean:  0.04, std: 0.94


In [7]:
# 8) Save processed data and scaler
PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

MODELS_DIR = Path('..') / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')

print("Saved processed data to data/processed/")
print("Saved scaler to models/scaler.pkl")
print(f"\nFeature names: {list(X_train.columns)}")
print(f"Final X_train shape: {X_train.shape}")
print(f"Final X_test shape: {X_test.shape}")

Saved processed data to data/processed/
Saved scaler to models/scaler.pkl

Feature names: ['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE', 'CHRONIC DISEASE', 'FATIGUE', 'ALLERGY', 'WHEEZING', 'ALCOHOL CONSUMING', 'COUGHING', 'SHORTNESS OF BREATH', 'SWALLOWING DIFFICULTY', 'CHEST PAIN']
Final X_train shape: (447, 15)
Final X_test shape: (112, 15)
